# ============================================================
# SECTION 2: TRANSFORMASI ECOCROP & PHYSICS CONSTRAINT
# ============================================================
# Jalankan setelah Section 1
# ============================================================

In [2]:
import pickle
import numpy as np
import tensorflow as tf

with open('session_preprocessing.pkl', 'rb') as f:
    sv = pickle.load(f)

le             = sv['le']
scaler         = sv['scaler']
ecocrop        = sv['ecocrop']
X_train_scaled = sv['X_train_scaled']
X_val_scaled   = sv['X_val_scaled']
X_test_scaled  = sv['X_test_scaled']
y_train_oh     = sv['y_train_oh']
y_val_oh       = sv['y_val_oh']
y_test_oh      = sv['y_test_oh']

print("✅ Preprocessing dimuat")

✅ Preprocessing dimuat


In [9]:
# ------------------------------------------------------------
# 2.1 Transformasi ECOCROP Min-Max → μ dan σ (Persamaan 3.1 & 3.2)
# ------------------------------------------------------------
CLASS_NAMES  = list(le.classes_)
TEMP_IDX     = 3
PH_IDX       = 5

physics_mu    = np.zeros((22, 2))
physics_sigma = np.zeros((22, 2))

for i, crop in enumerate(CLASS_NAMES):
    for j, feat in enumerate(['temperature', 'ph']):
        min_val = ecocrop[crop][feat][0]
        max_val = ecocrop[crop][feat][1]
        physics_mu[i, j]    = (max_val + min_val) / 2   # Persamaan 3.1
        physics_sigma[i, j] = (max_val - min_val) / 6   # Persamaan 3.2

print("Contoh mu rice (index 20)   :", physics_mu[20])
print("Contoh sigma rice (index 20):", physics_sigma[20])

Contoh mu rice (index 20)   : [25.    6.25]
Contoh sigma rice (index 20): [1.66666667 0.25      ]


In [10]:
# ------------------------------------------------------------
# 2.2 Scale μ dan σ ke ruang StandardScaler
# ------------------------------------------------------------
scaler_mean  = scaler.mean_
scaler_std   = scaler.scale_
feat_indices = [TEMP_IDX, PH_IDX]

physics_mu_scaled    = (physics_mu - scaler_mean[feat_indices]) / scaler_std[feat_indices]
physics_sigma_scaled = physics_sigma / scaler_std[feat_indices]

physics_mu_tf    = tf.constant(physics_mu_scaled,    dtype=tf.float32)
physics_sigma_tf = tf.constant(physics_sigma_scaled, dtype=tf.float32)

print("\nphysics_mu_tf shape   :", physics_mu_tf.shape)
print("physics_sigma_tf shape:", physics_sigma_tf.shape)
print("Contoh mu rice scaled  :", physics_mu_tf[20].numpy())
print("Contoh sigma rice scaled:", physics_sigma_tf[20].numpy())


physics_mu_tf shape   : (22, 2)
physics_sigma_tf shape: (22, 2)
Contoh mu rice scaled  : [-0.11379283 -0.2832632 ]
Contoh sigma rice scaled: [0.32526657 0.32223874]


In [12]:
# ------------------------------------------------------------
# 2.3 Verifikasi loss functions dengan data dummy
# ------------------------------------------------------------
def data_loss_fn(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
    return -tf.reduce_mean(tf.reduce_sum(y_true * tf.math.log(y_pred), axis=1))


def physics_loss_fn(X_batch, y_pred):
    temp = X_batch[:, 3:4]
    ph = X_batch[:, 5:6]
    mu_t = physics_mu_tf[:, 0]
    mu_p = physics_mu_tf[:, 1]
    sig_t = physics_sigma_tf[:, 0]
    sig_p = physics_sigma_tf[:, 1]
    temp_score = tf.exp(-tf.square(temp - mu_t) / (2 * tf.square(sig_t)))
    ph_score = tf.exp(-tf.square(ph - mu_p) / (2 * tf.square(sig_p)))
    env_score = temp_score * ph_score
    return tf.reduce_mean((1 - y_pred) * env_score)


test_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(7,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(22, activation='softmax')
])

X_sample = tf.constant(X_train_scaled[:32], dtype=tf.float32)
y_sample = tf.constant(y_train_oh[:32],     dtype=tf.float32)
y_pred_sample = test_model(X_sample, training=False)

print("\nVerifikasi loss functions:")
print("Data loss   :", data_loss_fn(y_sample, y_pred_sample).numpy().round(4))
print("Physics loss:", physics_loss_fn(
    X_sample, y_pred_sample).numpy().round(4))
print("Keduanya harus positif dan tidak NaN")

print("\n✅ [SECTION 2 SELESAI] Transformasi ECOCROP terverifikasi")


Verifikasi loss functions:
Data loss   : 3.0666
Physics loss: 0.1025
Keduanya harus positif dan tidak NaN

✅ [SECTION 2 SELESAI] Transformasi ECOCROP terverifikasi
